## Project 4: Predictive Analysis using scikit-learn

In this project, I am continuing from the mushroom preprocessing assignment. The goal is to use scikit-learn to determine which predictor column does a better job of predicting whether a mushroom is edible or poisonous.

The two predictor columns I selected are:

1. odor  
2. cap_color  

The target column is edible_poisonous, where:

0 = edible  
1 = poisonous  

Since odor and cap color are categorical values, I will use one-hot encoding with pandas get_dummies() before applying the machine learning models.

## Loading and Preparing the Dataset

This section recreates the preprocessing steps from the previous assignment.

In [11]:
import pandas as pd

df = pd.read_csv("agaricus-lepiota.data", header=None)

columns = [
    "class", "cap_shape", "cap_surface", "cap_color", "bruises", "odor",
    "gill_attachment", "gill_spacing", "gill_size", "gill_color",
    "stalk_shape", "stalk_root", "stalk_surface_above_ring",
    "stalk_surface_below_ring", "stalk_color_above_ring",
    "stalk_color_below_ring", "veil_type", "veil_color",
    "ring_number", "ring_type", "spore_print_color",
    "population", "habitat"
]

df.columns = columns

df_subset = df[["class", "odor", "cap_color"]].copy()

df_subset.columns = ["edible_poisonous", "odor", "cap_color"]

df_subset["edible_poisonous"] = df_subset["edible_poisonous"].map({
    "e": 0,
    "p": 1
})

odor_map = {
    "a": 0, "l": 1, "c": 2, "y": 3,
    "f": 4, "m": 5, "n": 6, "p": 7, "s": 8
}

color_map = {
    "n": 0, "b": 1, "c": 2, "g": 3,
    "r": 4, "p": 5, "u": 6, "e": 7,
    "w": 8, "y": 9
}

df_subset["odor"] = df_subset["odor"].map(odor_map)
df_subset["cap_color"] = df_subset["cap_color"].map(color_map)

print("Prepared dataset:")
print(df_subset.head())

Prepared dataset:
   edible_poisonous  odor  cap_color
0                 1     7          0
1                 0     0          9
2                 0     1          8
3                 1     7          8
4                 0     6          3


## Reviewing the Prepared Dataset

Before moving into modeling, I am taking another look at the cleaned dataset to confirm that all columns are in the correct format and ready for use.

This step helps verify that:
- the target variable has been properly encoded  
- the predictor columns are numeric  
- the dataset structure looks correct overall  

In [12]:
print("Preview of the cleaned dataset:")
print(df_subset.head())

print("\nData types:")
print(df_subset.dtypes)

print("\nShape of dataset:")
print(df_subset.shape)

Preview of the cleaned dataset:
   edible_poisonous  odor  cap_color
0                 1     7          0
1                 0     0          9
2                 0     1          8
3                 1     7          8
4                 0     6          3

Data types:
edible_poisonous    int64
odor                int64
cap_color           int64
dtype: object

Shape of dataset:
(8124, 3)


## Separating the Target Variable

Before building the models, I need to separate the target variable from the predictor variables.

The target variable is edible_poisonous because this is what the model is trying to predict.

In [13]:
y = df_subset["edible_poisonous"]

print("Target variable preview:")
print(y.head())

print("\nTarget value counts:")
print(y.value_counts())

Target variable preview:
0    1
1    0
2    0
3    1
4    0
Name: edible_poisonous, dtype: int64

Target value counts:
edible_poisonous
0    4208
1    3916
Name: count, dtype: int64


## Creating a Function to Test Each Predictor

To compare odor and cap color fairly, I will create a function that:

1. Selects one predictor column  
2. Converts it into one-hot encoded columns  
3. Splits the data into training and testing sets  
4. Trains a logistic regression model  
5. Prints the accuracy score  

This makes it easier to test both predictors using the same process.

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

def test_predictor(feature_name):
    print("\nTesting predictor:", feature_name)
    
    X = df_subset[[feature_name]]
    
    X_encoded = pd.get_dummies(
        X,
        columns=[feature_name],
        prefix=[feature_name]
    )
    
    print("\nEncoded feature preview:")
    print(X_encoded.head())
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_encoded,
        y,
        test_size=0.30,
        random_state=42,
        stratify=y
    )
    
    print("\nTraining set shape:", X_train.shape)
    print("Testing set shape:", X_test.shape)
    
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    
    print("\nAccuracy score:")
    print(accuracy)
    
    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    print("\nClassification report:")
    print(classification_report(y_test, y_pred))
    
    return accuracy

## Model 1: Odor as the Predictor

The first model will use odor by itself to predict whether a mushroom is edible or poisonous.

In [15]:
odor_accuracy = test_predictor("odor")


Testing predictor: odor

Encoded feature preview:
   odor_0  odor_1  odor_2  odor_3  odor_4  odor_5  odor_6  odor_7  odor_8
0   False   False   False   False   False   False   False    True   False
1    True   False   False   False   False   False   False   False   False
2   False    True   False   False   False   False   False   False   False
3   False   False   False   False   False   False   False    True   False
4   False   False   False   False   False   False    True   False   False

Training set shape: (5686, 9)
Testing set shape: (2438, 9)

Accuracy score:
0.9868744872846595

Confusion matrix:
[[1263    0]
 [  32 1143]]

Classification report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1263
           1       1.00      0.97      0.99      1175

    accuracy                           0.99      2438
   macro avg       0.99      0.99      0.99      2438
weighted avg       0.99      0.99      0.99      2438



## Model 2: Cap Color as the Predictor

The second model will use cap color by itself to predict whether a mushroom is edible or poisonous.

In [16]:
cap_color_accuracy = test_predictor("cap_color")


Testing predictor: cap_color

Encoded feature preview:
   cap_color_0  cap_color_1  cap_color_2  cap_color_3  cap_color_4  \
0         True        False        False        False        False   
1        False        False        False        False        False   
2        False        False        False        False        False   
3        False        False        False        False        False   
4        False        False        False         True        False   

   cap_color_5  cap_color_6  cap_color_7  cap_color_8  cap_color_9  
0        False        False        False        False        False  
1        False        False        False        False         True  
2        False        False        False         True        False  
3        False        False        False         True        False  
4        False        False        False        False        False  

Training set shape: (5686, 10)
Testing set shape: (2438, 10)

Accuracy score:
0.595980311730927

Confusion m

## Comparing the Two Predictors

Now I will compare the accuracy scores for odor and cap color.

In [17]:
print("Accuracy comparison:")
print("Odor accuracy:", odor_accuracy)
print("Cap color accuracy:", cap_color_accuracy)

if odor_accuracy > cap_color_accuracy:
    print("\nOdor is the better predictor.")
elif cap_color_accuracy > odor_accuracy:
    print("\nCap color is the better predictor.")
else:
    print("\nBoth predictors performed the same.")

Accuracy comparison:
Odor accuracy: 0.9868744872846595
Cap color accuracy: 0.595980311730927

Odor is the better predictor.


## Testing Both Predictors Together

Although the main goal is to compare the two predictors, I also want to test whether using both odor and cap color together improves the model.

In [18]:
X_both = df_subset[["odor", "cap_color"]]

X_both_encoded = pd.get_dummies(
    X_both,
    columns=["odor", "cap_color"],
    prefix=["odor", "cap_color"]
)

print("Encoded dataset using both predictors:")
print(X_both_encoded.head())

X_train, X_test, y_train, y_test = train_test_split(
    X_both_encoded,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

model_both = LogisticRegression(max_iter=1000)
model_both.fit(X_train, y_train)

y_pred_both = model_both.predict(X_test)

both_accuracy = accuracy_score(y_test, y_pred_both)

print("\nAccuracy using both odor and cap color:")
print(both_accuracy)

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred_both))

print("\nClassification report:")
print(classification_report(y_test, y_pred_both))

Encoded dataset using both predictors:
   odor_0  odor_1  odor_2  odor_3  odor_4  odor_5  odor_6  odor_7  odor_8  \
0   False   False   False   False   False   False   False    True   False   
1    True   False   False   False   False   False   False   False   False   
2   False    True   False   False   False   False   False   False   False   
3   False   False   False   False   False   False   False    True   False   
4   False   False   False   False   False   False    True   False   False   

   cap_color_0  cap_color_1  cap_color_2  cap_color_3  cap_color_4  \
0         True        False        False        False        False   
1        False        False        False        False        False   
2        False        False        False        False        False   
3        False        False        False        False        False   
4        False        False        False         True        False   

   cap_color_5  cap_color_6  cap_color_7  cap_color_8  cap_color_9  
0       

## Final Accuracy Comparison

In [19]:
results = pd.DataFrame({
    "Model": ["Odor Only", "Cap Color Only", "Odor and Cap Color"],
    "Accuracy": [odor_accuracy, cap_color_accuracy, both_accuracy]
})

print("Final model comparison:")
print(results)

Final model comparison:
                Model  Accuracy
0           Odor Only  0.986874
1      Cap Color Only  0.595980
2  Odor and Cap Color  0.986874


## Analysis

The final model comparison clearly shows how each predictor performs.

The odor-only model achieved an accuracy of 0.9869, which is extremely high.
The cap color model only reached 0.5960, which is much lower and indicates weak predictive ability.
The combined model (odor + cap color) also achieved 0.9869, which is exactly the same as odor alone.

This confirms that odor is doing nearly all of the predictive work. Cap color does not improve the model at all when added, which means it does not provide additional useful information beyond what odor already captures. The identical accuracy between the odor-only model and the combined model indicates that cap color does not contribute additional predictive value once odor is included.

## Conclusion

Based on the model comparison, odor is clearly the most effective predictor of whether a mushroom is poisonous. The odor-only model achieved an accuracy of approximately **98.7%**, while the cap color model only reached about **59.6%**. When both predictors were used together, the accuracy remained the same as odor alone, confirming that cap color does not add meaningful predictive value.

These results strongly align with the earlier exploratory analysis. When examining poison rates by odor, several odor categories had a poison rate of **1.0**, meaning all mushrooms with those odors were poisonous, while others had a rate of **0.0**, meaning they were always edible. There was also one category with a very low poison rate of approximately **0.034**, indicating only a small number of poisonous cases. This clear separation explains why odor performs so well in the model.

In contrast, cap color showed much more variation in poison rates, with values spread across a wide range and most categories containing a mix of both edible and poisonous mushrooms. This lack of clear separation makes cap color a weaker predictor, which is reflected in its lower model accuracy. Clearly, odor is the most reliable feature for predicting mushroom edibility in this dataset, and the modeling results confirm the patterns observed during the initial data exploration.

## Recommendations for Further Analysis

For further analysis, I would test additional mushroom features such as gill size, spore print color, bruises, and habitat. These columns may provide extra information that improves prediction accuracy.

I would also try other machine learning models, such as a decision tree classifier or random forest classifier. These models may work especially well because the dataset is categorical and contains clear decision patterns.

Finally, I would compare feature importance across multiple predictors to better understand which mushroom characteristics are most useful for identifying poisonous mushrooms.